# 금융 테크 블로그 → ChromaDB 벡터 저장소 구축
**Runtime → Change runtime type → T4 GPU** 로 설정 후 실행하세요.

In [ ]:
# ── 1. 패키지 설치 ────────────────────────────────────────────────────────────
!pip install -q langchain langchain-huggingface langchain-chroma chromadb sentence-transformers

In [ ]:
# 기존에 꼬였을 수 있는 패키지를 깔끔하게 다시 설치합니다.
!pip install -U langchain langchain-community langchain-core

In [52]:
# ── 2. GPU 확인 ───────────────────────────────────────────────────────────────
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 모델:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU 사용 가능: True
GPU 모델: Tesla T4
VRAM: 15.6 GB


In [53]:
# ── 3. finance_tech_content.json 업로드 ──────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # finance_tech_content.json 선택
print('업로드된 파일:', list(uploaded.keys()))

Saving finance_tech_content.json to finance_tech_content (3).json
업로드된 파일: ['finance_tech_content (3).json']


In [54]:
import json, re
from datetime import datetime
from urllib.parse import urlparse
from langchain_core.documents import Document
# 1. 랭체인 최신 스플리터 임포트
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 설정값
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 100

# [기존 기능 유지] 텍스트 정제 함수
def clean_text(text):
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'\[[^\]]{1,30}\]', '', text)
    text = re.sub(r'\d{4}[년.\-]\s*\d{1,2}[월.\-]\s*\d{1,2}일?', '', text)
    text = re.sub(r'https?://\S+', '', text)
    for n in ['공유하기','복사하기','카카오톡','트위터','페이스북','링크드인']:
        text = text.replace(n, '')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# [기존 기능 유지] 날짜 추출 함수
_DATE_PAT = re.compile(r'(\d{4})[.\-년]\s*(\d{1,2})[.\-월]\s*(\d{1,2})')
def extract_date(text):
    m = _DATE_PAT.search(text)
    if m:
        try:
            datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)))
            return f"{m.group(1)}-{m.group(2).zfill(2)}-{m.group(3).zfill(2)}"
        except ValueError:
            pass
    return ''

# [수정된 핵심 함수] 문서 빌드 및 정밀 분할
def build_documents(data):
    docs = []

    # 2. 정밀 절단기(Splitter) 설정
    # 단어 중간이 잘리지 않도록 마침표, 줄바꿈, 공백 등을 기준으로 영리하게 자릅니다.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " ", ""], # 자르는 우선순위
        is_separator_regex=False,
    )

    for entry in data:
        url, title, content = entry.get('url',''), entry.get('title',''), entry.get('content','')

        # 텍스트 정제
        cleaned  = clean_text(content)

        # 메타데이터 추출 (날짜 및 도메인)
        pub_date = extract_date(cleaned) or extract_date(content)
        try:
            domain = urlparse(url).hostname.removeprefix('www.')
        except:
            domain = ''

        # 3. 랭체인 스플리터를 사용해 텍스트 분할
        chunks = text_splitter.split_text(cleaned)

        for i, chunk in enumerate(chunks):
            # 의미 있는 길이(30자 이상)의 조각만 문서로 만듭니다.
            if len(chunk) > 30:
                docs.append(Document(
                    page_content=chunk,
                    metadata={
                        'source': url,
                        'title': title,
                        'domain': domain,
                        'pub_date': pub_date,
                        'chunk_id': i
                    }
                ))
    return docs

print('✅ 수정된 전처리 함수 정의 완료 (단어 잘림 방지 로직 적용)')

✅ 수정된 전처리 함수 정의 완료 (단어 잘림 방지 로직 적용)


In [55]:
# ── 5. 데이터 로드 및 청킹 ────────────────────────────────────────────────────
with open('finance_tech_content.json', encoding='utf-8') as f:
    data = json.load(f)

docs = build_documents(data)
print(f'아티클 {len(data)}개 → 청크 {len(docs)}개')

아티클 568개 → 청크 9071개


In [56]:
# ── 6. 임베딩 모델 로드 (BAAI/bge-m3, T4 GPU) ────────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {DEVICE}')

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': DEVICE},
    encode_kwargs={'batch_size': 64, 'normalize_embeddings': True},
)
print('모델 로드 완료')

디바이스: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

모델 로드 완료


In [59]:
# ── 7. ChromaDB 저장 ──────────────────────────────────────────────────────────
from langchain_chroma import Chroma

CHROMA_DIR = '/content/chroma_db1'

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name='finance_tech',
    persist_directory=CHROMA_DIR,
)
print(f'ChromaDB 저장 완료 → {CHROMA_DIR}')

ChromaDB 저장 완료 → /content/chroma_db1


In [60]:
# ── 8. 검색 테스트 ────────────────────────────────────────────────────────────
results = vectorstore.similarity_search('결제 정산 시스템', k=3)
for i, r in enumerate(results, 1):
    print(f'\n[{i}] {r.metadata["title"]} ({r.metadata["domain"]})')
    print(r.page_content[:200])


[1] 레거시 정산 개편기: 신규 시스템 투입 여정부터 대규모 배치 운영 노하우까지 (toss.tech)
레거시 정산 개편기: 신규 시스템 투입 여정부터 대규모 배치 운영 노하우까지
강민주/박진현

안녕하세요, 토스페이먼츠 Server Developer 강민주, 박진현입니다.
"정산 시스템을 왜 개편하나요?"
토스페이먼츠에서 정산 시스템을 운영하며 가장 많이 받았던 질문이에요. 하지만 20년이 넘은 레거시 시스템을 운영하다 보니, 저희가 반드시 극복해야 할 명

[2] Spring Batch를 더 우아하게 사용하기 - Spring Batch Plus (d2.naver.com)
네이버페이 정산 시스템은 금액 집계, 지급 요청 등 서로 다른 역할을 하는 수십 개의 Job으로 구성된다. 그런데 이 Job은 상호 의존한다. 이를테면 정산 금액을 집계하지도 않았는데 판매자에게 정산 금액을 지급할 수는 없는 일이다. Spring Batch는 Job 간 의존관계를 설정할 수 있는 JobStep 기능을 제공한다. JobStep을 사용하면 한 

[3] 레거시 정산 개편기: 신규 시스템 투입 여정부터 대규모 배치 운영 노하우까지 (toss.tech)
신규 시스템 기반으로도 정산 건을 생성합니다.
두 시스템이 만든 데이터를 기반으로 
두 데이터가 동일한지 검증
을 진행합니다.
두 시스템에서 만든 거래가 동일하면
 신규 시스템으로 만들어진 데이터를 투입합니다.
불일치하는 경우
에는 레거시 시스템으로 만들어진 데이터를 투입하고, 불일치의 원인을 이후에 트래킹합니다.
세밀한 투입 단위
저희는 투입의 단위를 더


In [63]:
# ── 9. chroma_db 폴더를 zip으로 다운로드 ─────────────────────────────────────
import shutil
shutil.make_archive('/content/chroma_db1', 'zip', '/content/chroma_db1')
files.download('/content/chroma_db1.zip')
print('다운로드 완료 → chroma_db1.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드 완료 → chroma_db1.zip


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ 찾은 데이터 개수: 0개


✅ 데이터 개수: 8695개


In [ ]:
=

Mounted at /content/drive
